##### **Bronze Layer**
- ##### *Load every sheet of OLTP.xlsx (lakehouse Files/raw) → dbo.<sheet>_raw*
- Replaces the old Dataflow Gen2 step: no gateway or OAuth connection needed, and it is re-runnable.

In [ ]:
# Parameters (can be overridden from the pipeline)
source_path = "/lakehouse/default/Files/raw/OLTP.xlsx"

In [ ]:
import pandas as pd

SHEETS = ["region", "geography", "channels", "order_statuses", "customers", "customer_segments",
          "customer_addresses", "products", "orders", "order_items", "dates"]

book = pd.read_excel(source_path, sheet_name=SHEETS, engine="openpyxl")

loaded = {}
for sheet, pdf in book.items():
    # Excel blanks arrive as NaN; turn them into real NULLs so Spark infers clean types
    for col in pdf.columns[pdf.isna().any()]:
        pdf[col] = pdf[col].astype(object).where(pdf[col].notna(), None)
    (spark.createDataFrame(pdf)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"dbo.{sheet}_raw"))
    loaded[sheet] = len(pdf)

print(loaded)
empty = [s for s, n in loaded.items() if n == 0]
if empty:
    raise ValueError(f"Bronze load failed - empty sheets: {empty}")